# PS2: Right-Party Contact Prediction (Demo/Lightweight Pipeline)

This notebook builds a lightweight pipeline to predict whether a contact number is valid and reaches the right party.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import os

## 2. Data Loading

In [ ]:
base_path = r"../../Dataset/data"

In [ ]:
phones = pd.read_csv(os.path.join(base_path, "ps2_right_party_contact", "phones.csv"))
dial_attempts = pd.read_csv(os.path.join(base_path, "shared", "dial_attempts.csv"))
verified = pd.read_csv(os.path.join(base_path, "ps2_right_party_contact", "verified_contact_points.csv"))

## 3. Data Preprocessing & Target Definition

In [ ]:
verified['is_rpc'] = (verified['verified_status'] == 'borrower_number').astype(int)

In [ ]:
attempts_agg = dial_attempts.groupby('phone_id').agg(
    total_attempts=('attempt_id', 'count'),
    avg_ring_duration=('ring_duration_s', 'mean'),
    total_talk_time=('talk_duration_s', 'sum')
).reset_index()

In [ ]:
data = pd.merge(verified, phones, on='phone_id', how='left')
data = pd.merge(data, attempts_agg, on='phone_id', how='left').fillna({'total_attempts': 0, 'avg_ring_duration': 0, 'total_talk_time': 0})

In [ ]:
features = ['priority_slot', 'total_attempts', 'avg_ring_duration', 'total_talk_time']
X = data[features]
y = data['is_rpc']

## 4. Model Training and Evaluation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
clf = RandomForestClassifier(n_estimators=50, random_state=42, max_depth=3)
clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

In [ ]:
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
print(f"ROC AUC Score: {roc_auc_score(y_test, y_prob):.4f}")